# InvariantRRF v4.3 - Partial-Observation Closure Audit

This notebook **does not retrain or rerun any retriever**.

It imports the completed v4.3 canonical results archive and recomputes the descriptive partial-observation quantities used in manuscript Section 6.7:

- top-1 flip rate;
- exact ordered top-10 reproduction;
- exact top-10 set reproduction as an additional audit field;
- all prespecified observation depths;
- BM25+dense on SciFact, TREC-COVID, FiQA, and ArguAna;
- BM25+SPLADE+dense on SciFact and ArguAna.

### Canonical references

Two-source reference:

- BM25 depth = 1000
- dense depth = 1000
- ordinary RRF `k = 60`

Three-source reference:

- BM25 depth = 1000
- SPLADE depth = 500
- dense depth = 1000
- ordinary RRF `k = 60`

Requested shallow depths are clipped to each source's generated maximum.

The notebook reads the canonical ranked-list JSON files, verifies their integrity, optionally verifies the v4.3 manifest, computes all query-level outcomes, and writes a small paper-closure ZIP containing the new source-truth tables and manuscript-ready text.


In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import shutil
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# ============================================================
# USER CONFIGURATION
# ============================================================

# Preferred: upload/add the completed v4.3 results ZIP to the notebook.
# If None, this notebook auto-discovers a likely InvariantRRF canonical ZIP.
#
# Kaggle example:
# INPUT_ZIP = "/kaggle/input/my-v43-results/InvariantRRF_Canonical_STRICT_DeepDense_TaskAdaptiveK_Results.zip"
#
# Colab/local example:
# INPUT_ZIP = "/content/InvariantRRF_Canonical_STRICT_DeepDense_TaskAdaptiveK_Results.zip"
INPUT_ZIP = None

# Alternative: if you already extracted the v4.3 canonical directory,
# set this to the directory containing runs/, tables/, statistics/, etc.
CANON_OVERRIDE = None

RRF_K = 60.0
OUTPUT_K = 10
DEPTH_GRID = [5, 10, 20, 30, 50, 75, 100, 150, 200, 300, 500, 750, 1000]

EXPECTED_Q = {
    "SciFact": 300,
    "TREC-COVID": 50,
    "FiQA": 648,
    "ArguAna": 1401,
}

WORK_ROOT = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path("/content")
    if Path("/content").exists()
    else Path.cwd()
)

CLOSURE_DIR = WORK_ROOT / "InvariantRRF_V43_PartialObservation_Closure"
if CLOSURE_DIR.exists():
    shutil.rmtree(CLOSURE_DIR)
CLOSURE_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT:", WORK_ROOT)
print("CLOSURE_DIR:", CLOSURE_DIR)


In [ ]:
# ============================================================
# 1. Locate/extract the completed v4.3 canonical package
# ============================================================

def _zip_priority(path):
    name = path.name.lower()
    score = 0
    if "invariantrrf" in name:
        score += 100
    if "deepdense" in name:
        score += 50
    if "taskadaptivek" in name:
        score += 40
    if "canonical" in name:
        score += 30
    if "strict" in name:
        score += 20
    if "fourdataset" in name:
        score += 10
    return score


def discover_zip():
    roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path("/content"),
        Path.cwd(),
    ]
    candidates = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*.zip"):
                rp = str(p.resolve())
                if rp in seen:
                    continue
                seen.add(rp)
                candidates.append(p)
        except Exception:
            pass

    invariant = [p for p in candidates if "invariantrrf" in p.name.lower()]
    pool = invariant if invariant else candidates

    if not pool:
        raise FileNotFoundError(
            "No ZIP archive found. Set INPUT_ZIP to the completed v4.3 results archive."
        )

    pool = sorted(pool, key=lambda p: (_zip_priority(p), str(p)), reverse=True)
    print("ZIP candidates:")
    for p in pool[:10]:
        print(" ", p, "priority=", _zip_priority(p))
    return pool[0]


def find_canonical_root(root):
    root = Path(root)

    if (root / "runs").is_dir():
        return root

    manifests = sorted(root.rglob("RUN_MANIFEST.json"))
    for manifest in manifests:
        parent = manifest.parent
        if (parent / "runs").is_dir():
            return parent

    run_dirs = sorted(p for p in root.rglob("runs") if p.is_dir())
    for run_dir in run_dirs:
        parent = run_dir.parent
        if (parent / "runs").is_dir():
            return parent

    raise FileNotFoundError(
        f"Could not find canonical directory containing runs/ under {root}"
    )


if CANON_OVERRIDE is not None:
    CANON = find_canonical_root(Path(CANON_OVERRIDE))
    SOURCE_ARCHIVE = None
else:
    archive = Path(INPUT_ZIP) if INPUT_ZIP else discover_zip()
    if not archive.exists():
        raise FileNotFoundError(archive)

    SOURCE_ARCHIVE = archive
    EXTRACT_ROOT = WORK_ROOT / "_v43_imported_results"

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive, "r") as z:
        z.extractall(EXTRACT_ROOT)

    CANON = find_canonical_root(EXTRACT_ROOT)

CANON_RUNS = CANON / "runs"

print("\nSOURCE_ARCHIVE:", SOURCE_ARCHIVE)
print("CANON:", CANON)
print("CANON_RUNS:", CANON_RUNS)
assert CANON_RUNS.is_dir()


In [ ]:
# ============================================================
# 2. Verify original package manifest when available
# ============================================================

def sha256_file(path, chunk=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


manifest_path = CANON / "RUN_MANIFEST.json"

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    files = manifest.get("files", {})
    failures = []
    checked = 0

    for rel, expected in files.items():
        p = CANON / rel
        if not p.exists():
            failures.append((rel, "MISSING", expected))
            continue

        actual = sha256_file(p)
        checked += 1
        if actual != expected:
            failures.append((rel, actual, expected))

    if failures:
        print("Manifest failures:")
        for x in failures[:20]:
            print(x)
        raise AssertionError(
            f"Original v4.3 manifest verification failed for {len(failures)} file(s)."
        )

    print(f"ORIGINAL RUN MANIFEST: PASS ({checked} files verified)")
else:
    manifest = None
    print("WARNING: RUN_MANIFEST.json not present; continuing with direct run-file integrity checks.")


In [ ]:
# ============================================================
# 3. Load and validate the exact canonical ranked-list artifacts
# ============================================================

REQUIRED_TWO_SOURCE = {
    ds: {
        "bm25": CANON_RUNS / f"{ds}_bm25_top1000.json",
        "dense": CANON_RUNS / f"{ds}_dense_STRICT_top1000.json",
    }
    for ds in EXPECTED_Q
}

REQUIRED_THREE_SOURCE = {
    "SciFact": {
        "bm25": CANON_RUNS / "SciFact_bm25_top1000.json",
        "splade": CANON_RUNS / "SciFact_splade_ensemble_top500.json",
        "dense": CANON_RUNS / "SciFact_dense_STRICT_top1000.json",
    },
    "ArguAna": {
        "bm25": CANON_RUNS / "ArguAna_bm25_top1000.json",
        "splade": CANON_RUNS / "ArguAna_splade_ensemble_top500.json",
        "dense": CANON_RUNS / "ArguAna_dense_STRICT_top1000.json",
    },
}


def load_ranked_run(path):
    path = Path(path)
    assert path.exists(), f"Missing canonical run: {path}"

    obj = json.loads(path.read_text(encoding="utf-8"))
    assert isinstance(obj, dict) and obj, f"Invalid or empty run: {path}"

    normalized = {}
    for qid, seq in obj.items():
        if not isinstance(seq, list):
            raise TypeError(f"{path.name}/{qid}: ranking is not a list")
        normalized[str(qid)] = seq

    return normalized


def docids(seq):
    ids = [
        str(x[0]) if isinstance(x, (list, tuple)) else str(x)
        for x in seq
    ]
    if len(ids) != len(set(ids)):
        raise AssertionError("Ranking contains duplicate document IDs.")
    return ids


def validate_run(run, expected_queries, min_depth, label):
    assert len(run) == expected_queries, (
        label, "query count", len(run), "expected", expected_queries
    )

    depths = []
    for qid, seq in run.items():
        ids = docids(seq)
        depths.append(len(ids))
        if len(ids) < min_depth:
            raise AssertionError(
                f"{label}/{qid}: depth {len(ids)} < required {min_depth}"
            )

    return {
        "label": label,
        "queries": len(run),
        "min_depth": min(depths),
        "max_depth": max(depths),
    }


source_runs = {}
source_rows = []
source_hashes = {}

for dataset, paths in REQUIRED_TWO_SOURCE.items():
    source_runs.setdefault(dataset, {})
    for source, path in paths.items():
        run = load_ranked_run(path)
        source_runs[dataset][source] = run
        source_rows.append(
            validate_run(
                run,
                EXPECTED_Q[dataset],
                1000,
                f"{dataset}/{source}",
            )
        )
        source_hashes[str(path.relative_to(CANON))] = sha256_file(path)

for dataset, paths in REQUIRED_THREE_SOURCE.items():
    for source, path in paths.items():
        if source in source_runs.get(dataset, {}):
            continue

        run = load_ranked_run(path)
        source_runs[dataset][source] = run
        min_depth = 500 if source == "splade" else 1000
        source_rows.append(
            validate_run(
                run,
                EXPECTED_Q[dataset],
                min_depth,
                f"{dataset}/{source}",
            )
        )
        source_hashes[str(path.relative_to(CANON))] = sha256_file(path)

source_df = pd.DataFrame(source_rows).sort_values("label").reset_index(drop=True)
display(source_df)

print("\nCANONICAL RUN FILES: PASS")


In [ ]:
# ============================================================
# 4. Ordinary RRF and partial-observation audit
# ============================================================

def rrf_order(runs, qid, depths, k=60.0):
    # Ordinary RRF with uniform source weights.
    # Deterministic final order: descending RRF score, then ascending document ID.
    scores = defaultdict(float)

    for source, run in runs.items():
        ids = docids(run[str(qid)])
        L = min(int(depths[source]), len(ids))

        for rank, did in enumerate(ids[:L], start=1):
            scores[did] += 1.0 / (float(k) + rank)

    return [
        did
        for did, _ in sorted(
            scores.items(),
            key=lambda kv: (-kv[1], kv[0]),
        )
    ]


def evaluate_configuration(dataset, configuration, runs, reference_depths):
    qsets = [set(run.keys()) for run in runs.values()]
    qids = sorted(set.intersection(*qsets))

    assert len(qids) == EXPECTED_Q[dataset], (
        dataset,
        configuration,
        len(qids),
        EXPECTED_Q[dataset],
    )

    rows = []

    for qid in qids:
        reference = rrf_order(
            runs,
            qid,
            reference_depths,
            k=RRF_K,
        )[:OUTPUT_K]

        assert len(reference) == OUTPUT_K, (
            dataset,
            configuration,
            qid,
            len(reference),
        )

        for requested_depth in DEPTH_GRID:
            observed_depths = {
                source: min(
                    int(requested_depth),
                    int(reference_depths[source]),
                    len(runs[source][qid]),
                )
                for source in runs
            }

            observed = rrf_order(
                runs,
                qid,
                observed_depths,
                k=RRF_K,
            )[:OUTPUT_K]

            ordered_match = (
                len(observed) == OUTPUT_K
                and observed == reference
            )

            set_match = (
                len(observed) == OUTPUT_K
                and set(observed) == set(reference)
            )

            top1_flip = (
                len(observed) == 0
                or observed[0] != reference[0]
            )

            rows.append({
                "dataset": dataset,
                "configuration": configuration,
                "qid": str(qid),
                "requested_depth": int(requested_depth),
                "observed_depths": json.dumps(observed_depths, sort_keys=True),
                "reference_depths": json.dumps(reference_depths, sort_keys=True),
                "top1_flip": bool(top1_flip),
                "ordered_top10_reproduction": bool(ordered_match),
                "set_top10_reproduction": bool(set_match),
            })

    return rows


audit_rows = []

for dataset in ["SciFact", "TREC-COVID", "FiQA", "ArguAna"]:
    runs = {
        "bm25": source_runs[dataset]["bm25"],
        "dense": source_runs[dataset]["dense"],
    }

    audit_rows.extend(
        evaluate_configuration(
            dataset=dataset,
            configuration="BM25+dense",
            runs=runs,
            reference_depths={"bm25": 1000, "dense": 1000},
        )
    )

for dataset in ["SciFact", "ArguAna"]:
    runs = {
        "bm25": source_runs[dataset]["bm25"],
        "splade": source_runs[dataset]["splade"],
        "dense": source_runs[dataset]["dense"],
    }

    audit_rows.extend(
        evaluate_configuration(
            dataset=dataset,
            configuration="BM25+SPLADE+dense",
            runs=runs,
            reference_depths={"bm25": 1000, "splade": 500, "dense": 1000},
        )
    )


per_query = pd.DataFrame(audit_rows)

summary = (
    per_query
    .groupby(["dataset", "configuration", "requested_depth"], as_index=False)
    .agg(
        queries=("qid", "nunique"),
        top1_flip_count=("top1_flip", "sum"),
        top1_flip_rate=("top1_flip", "mean"),
        ordered_top10_reproduction_count=("ordered_top10_reproduction", "sum"),
        ordered_top10_reproduction_rate=("ordered_top10_reproduction", "mean"),
        set_top10_reproduction_count=("set_top10_reproduction", "sum"),
        set_top10_reproduction_rate=("set_top10_reproduction", "mean"),
    )
)

for col in [
    "top1_flip_count",
    "ordered_top10_reproduction_count",
    "set_top10_reproduction_count",
]:
    summary[col] = summary[col].astype(int)

summary = summary.sort_values(
    ["configuration", "dataset", "requested_depth"]
).reset_index(drop=True)

expected_per_query_rows = (sum(EXPECTED_Q.values()) + EXPECTED_Q["SciFact"] + EXPECTED_Q["ArguAna"]) * len(DEPTH_GRID)
expected_summary_rows = (4 + 2) * len(DEPTH_GRID)
assert len(per_query) == expected_per_query_rows, (len(per_query), expected_per_query_rows)
assert len(summary) == expected_summary_rows, (len(summary), expected_summary_rows)

# At requested depth 1000, every source is at its deepest generated reference
# (SPLADE is clipped to 500), so reproduction must be exact by construction.
deep_rows = summary[summary["requested_depth"] == 1000]
assert len(deep_rows) == 6
assert (deep_rows["ordered_top10_reproduction_rate"] == 1.0).all()
assert (deep_rows["set_top10_reproduction_rate"] == 1.0).all()
assert (deep_rows["top1_flip_rate"] == 0.0).all()
print("DEEPEST-REFERENCE SELF-CHECK: PASS")

print("Per-query rows:", len(per_query))
print("Summary rows:", len(summary))

display(
    summary[
        summary["requested_depth"].isin([5, 50, 100])
    ]
)


In [ ]:
# ============================================================
# 5. Manuscript-facing values and stale-number comparison
# ============================================================

def get_row(configuration, dataset, depth):
    x = summary[
        (summary["configuration"] == configuration)
        & (summary["dataset"] == dataset)
        & (summary["requested_depth"] == depth)
    ]
    assert len(x) == 1, (configuration, dataset, depth)
    return x.iloc[0]


def pct(value):
    p = 100.0 * float(value)
    if abs(p) < 1e-15:
        return "0%"
    if p < 1.0:
        return f"{p:.2f}%"
    return f"{p:.1f}%"


datasets4 = ["SciFact", "TREC-COVID", "FiQA", "ArguAna"]

depth5_order = {
    ds: get_row("BM25+dense", ds, 5)["ordered_top10_reproduction_rate"]
    for ds in datasets4
}
depth5_top1 = {
    ds: get_row("BM25+dense", ds, 5)["top1_flip_rate"]
    for ds in datasets4
}
depth50_order = {
    ds: get_row("BM25+dense", ds, 50)["ordered_top10_reproduction_rate"]
    for ds in datasets4
}
depth100_order = {
    ds: get_row("BM25+dense", ds, 100)["ordered_top10_reproduction_rate"]
    for ds in datasets4
}
splade50 = {
    ds: get_row("BM25+SPLADE+dense", ds, 50)["ordered_top10_reproduction_rate"]
    for ds in ["SciFact", "ArguAna"]
}
splade100 = {
    ds: get_row("BM25+SPLADE+dense", ds, 100)["ordered_top10_reproduction_rate"]
    for ds in ["SciFact", "ArguAna"]
}

paragraph_lines = [
    (
        "The second half of the study holds source identities fixed and changes only how deeply "
        "their rankings are observed. Against the deepest generated BM25+dense fusion "
        "(1000 ranks from each source), a depth-5 fusion exactly reproduces the ordered top-10 "
        f'on {pct(depth5_order["SciFact"])} of SciFact queries, '
        f'{pct(depth5_order["TREC-COVID"])} of TREC-COVID queries, '
        f'{pct(depth5_order["FiQA"])} of FiQA queries, and '
        f'{pct(depth5_order["ArguAna"])} of ArguAna queries. '
        "The corresponding top-1 flip rates are "
        f'{pct(depth5_top1["SciFact"])}, {pct(depth5_top1["TREC-COVID"])}, '
        f'{pct(depth5_top1["FiQA"])}, and {pct(depth5_top1["ArguAna"])}. '
        "ArguAna therefore again shows why a stable first position does not imply a stable top-10 order."
    ),
    (
        "Deeper observation improves empirical reproduction at collection-specific rates. "
        "At depth 50, exact ordered top-10 reproduction is "
        f'{pct(depth50_order["SciFact"])} on SciFact, '
        f'{pct(depth50_order["TREC-COVID"])} on TREC-COVID, '
        f'{pct(depth50_order["FiQA"])} on FiQA, and '
        f'{pct(depth50_order["ArguAna"])} on ArguAna. '
        "At depth 100 the rates are "
        f'{pct(depth100_order["SciFact"])}, {pct(depth100_order["TREC-COVID"])}, '
        f'{pct(depth100_order["FiQA"])}, and {pct(depth100_order["ArguAna"])}. '
        "In the three-source BM25+SPLADE+dense configurations, depth-50 ordered reproduction is "
        f'{pct(splade50["SciFact"])} on SciFact and {pct(splade50["ArguAna"])} on ArguAna; '
        "at depth 100 the rates rise to "
        f'{pct(splade100["SciFact"])} and {pct(splade100["ArguAna"])}. '
        "These are empirical agreement rates with deeper generated runs, not certificates."
    ),
]

paper_paragraph = "\n\n".join(paragraph_lines)
display(Markdown("## Regenerated Section 6.7 text\n\n" + paper_paragraph))

stale = {
    ("BM25+dense", "SciFact", 5, "ordered"): (0.0000, "0%"),
    ("BM25+dense", "TREC-COVID", 5, "ordered"): (0.0200, "2.0%"),
    ("BM25+dense", "FiQA", 5, "ordered"): (0.0015, "0.15%"),
    ("BM25+dense", "ArguAna", 5, "ordered"): (0.0000, "0%"),

    ("BM25+dense", "SciFact", 5, "top1_flip"): (0.2100, "21.0%"),
    ("BM25+dense", "TREC-COVID", 5, "top1_flip"): (0.7400, "74.0%"),
    ("BM25+dense", "FiQA", 5, "top1_flip"): (0.5350, "53.5%"),
    ("BM25+dense", "ArguAna", 5, "top1_flip"): (0.0014, "0.14%"),

    ("BM25+dense", "SciFact", 50, "ordered"): (0.2030, "20.3%"),
    ("BM25+dense", "TREC-COVID", 50, "ordered"): (0.0400, "4.0%"),
    ("BM25+dense", "FiQA", 50, "ordered"): (0.1250, "12.5%"),
    ("BM25+dense", "ArguAna", 50, "ordered"): (0.8010, "80.1%"),

    ("BM25+dense", "SciFact", 100, "ordered"): (0.4600, "46.0%"),
    ("BM25+dense", "TREC-COVID", 100, "ordered"): (0.2000, "20.0%"),
    ("BM25+dense", "FiQA", 100, "ordered"): (0.3830, "38.3%"),
    ("BM25+dense", "ArguAna", 100, "ordered"): (0.9430, "94.3%"),

    ("BM25+SPLADE+dense", "SciFact", 50, "ordered"): (0.1000, "10.0%"),
    ("BM25+SPLADE+dense", "ArguAna", 50, "ordered"): (0.7120, "71.2%"),
    ("BM25+SPLADE+dense", "SciFact", 100, "ordered"): (0.2430, "24.3%"),
    ("BM25+SPLADE+dense", "ArguAna", 100, "ordered"): (0.8850, "88.5%"),
}

comparison_rows = []

for (configuration, dataset, depth, metric), (old_value, old_display) in stale.items():
    row = get_row(configuration, dataset, depth)

    if metric == "ordered":
        new_value = float(row["ordered_top10_reproduction_rate"])
    elif metric == "top1_flip":
        new_value = float(row["top1_flip_rate"])
    else:
        raise ValueError(metric)

    new_display = pct(new_value)
    comparison_rows.append({
        "configuration": configuration,
        "dataset": dataset,
        "depth": depth,
        "metric": metric,
        "stale_paper_display": old_display,
        "v43_regenerated_display": new_display,
        "stale_paper_nominal_rate": old_value,
        "v43_regenerated_rate": new_value,
        "difference_pp_vs_nominal": 100.0 * (new_value - old_value),
        "same_at_paper_precision": bool(new_display == old_display),
    })

comparison = pd.DataFrame(comparison_rows)

print("\nSTALE-PAPER COMPARISON")
display(comparison)

changed = int((~comparison["same_at_paper_precision"]).sum())
print(f"Changed stale entries: {changed}/{len(comparison)}")


In [ ]:
# ============================================================
# 6. Write closure artifacts
# ============================================================

per_query_path = CLOSURE_DIR / "canonical_partial_observation_per_query.csv"
summary_path = CLOSURE_DIR / "canonical_partial_observation_summary.csv"
comparison_path = CLOSURE_DIR / "stale_paper_vs_v43_regenerated.csv"
protocol_path = CLOSURE_DIR / "PARTIAL_OBSERVATION_PROTOCOL.json"
paper_md_path = CLOSURE_DIR / "PARTIAL_OBSERVATION_FOR_MANUSCRIPT.md"
paper_tex_path = CLOSURE_DIR / "PARTIAL_OBSERVATION_FOR_MANUSCRIPT.tex"
source_hash_path = CLOSURE_DIR / "SOURCE_RUN_HASHES.json"

per_query.to_csv(per_query_path, index=False)
summary.to_csv(summary_path, index=False)
comparison.to_csv(comparison_path, index=False)

protocol = {
    "analysis": "InvariantRRF v4.3 partial-observation closure audit",
    "source": "completed canonical v4.3 ranked-list artifacts",
    "retriever_retraining": False,
    "retriever_inference": False,
    "qrels_used": False,
    "rrf_k": RRF_K,
    "output_k": OUTPUT_K,
    "observation_depth_grid": DEPTH_GRID,
    "tie_break": "descending RRF score, then ascending document_id",
    "two_source_reference": {
        "configuration": "BM25+dense",
        "depths": {"bm25": 1000, "dense": 1000},
        "datasets": ["SciFact", "TREC-COVID", "FiQA", "ArguAna"],
    },
    "three_source_reference": {
        "configuration": "BM25+SPLADE+dense",
        "depths": {"bm25": 1000, "splade": 500, "dense": 1000},
        "datasets": ["SciFact", "ArguAna"],
    },
    "metrics": [
        "top1_flip",
        "exact_ordered_top10_reproduction",
        "exact_set_top10_reproduction",
    ],
    "stable_rrf_note": (
        "These are empirical reproduction rates against deepest generated fused runs, "
        "not StableRRF completion certificates. StableRRF dense cap 100 remains the "
        "designated baseline operating point only."
    ),
}

protocol_path.write_text(json.dumps(protocol, indent=2), encoding="utf-8")
source_hash_path.write_text(
    json.dumps(source_hashes, indent=2, sort_keys=True),
    encoding="utf-8",
)

paper_md = (
    "# InvariantRRF v4.3 partial-observation replacement text\n\n"
    "## Section 6.7\n\n"
    + paper_paragraph
    + "\n"
)
paper_md_path.write_text(paper_md, encoding="utf-8")

paper_tex = paper_paragraph.replace("%", r"\%")
paper_tex_path.write_text(paper_tex + "\n", encoding="utf-8")

print("Wrote:")
for p in [
    per_query_path,
    summary_path,
    comparison_path,
    protocol_path,
    source_hash_path,
    paper_md_path,
    paper_tex_path,
]:
    print(" ", p)


In [ ]:
# ============================================================
# 7. Build a self-verifying closure manifest and ZIP
# ============================================================

closure_manifest = {
    "source_archive": (
        str(SOURCE_ARCHIVE.name)
        if SOURCE_ARCHIVE is not None
        else None
    ),
    "source_run_hashes": source_hashes,
    "files": {},
}

for p in sorted(CLOSURE_DIR.iterdir()):
    if p.is_file() and p.name != "CLOSURE_MANIFEST.json":
        closure_manifest["files"][p.name] = sha256_file(p)

closure_manifest_path = CLOSURE_DIR / "CLOSURE_MANIFEST.json"
closure_manifest_path.write_text(
    json.dumps(closure_manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)

check = json.loads(closure_manifest_path.read_text(encoding="utf-8"))
for rel, expected in check["files"].items():
    actual = sha256_file(CLOSURE_DIR / rel)
    assert actual == expected, (rel, actual, expected)

print(
    "CLOSURE MANIFEST SELF-VERIFY: PASS",
    len(check["files"]),
    "files",
)

zip_out = WORK_ROOT / "InvariantRRF_V43_PartialObservation_Closure.zip"
if zip_out.exists():
    zip_out.unlink()

with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(CLOSURE_DIR.iterdir()):
        if p.is_file():
            z.write(p, arcname=f"{CLOSURE_DIR.name}/{p.name}")

print("\nFINAL CLOSURE ZIP:")
print(zip_out)

print("\nMANUSCRIPT TEXT:")
print(paper_paragraph)


## What to copy back into the paper

After the notebook finishes successfully:

1. Replace the **entire numerical text** in Section 6.7 with `PARTIAL_OBSERVATION_FOR_MANUSCRIPT.tex`.
2. Do not retain any old depth-5 / depth-50 / depth-100 reproduction numbers that disagree with the regenerated CSV.
3. Keep the current v4.3 Methods distinction:
   - BM25 and dense evidence are generated to depth 1000;
   - SPLADE is generated to depth 500 where used;
   - dense cap 100 is only the designated StableRRF baseline.
4. Add the closure ZIP to the reproducibility package.
5. If rebuilding the main canonical archive, rerun its manifest/packaging cell after copying these closure artifacts into the package.

No model training, dense inference, BM25 generation, or SPLADE generation is required by this notebook.
